# Smart Irrigation System Using Machine Learning

This notebook profiles the supplied workbook, then runs the reproducible baseline-versus-decision-tree workflow in `src/`. The workbook's `Notes` sheet defines `Best Time` as a time-window label; it is **not** an irrigation-need or pump-status label. The workbook also describes the data as semi-synthetic, including simulated soil-moisture and irrigation-target values. Treat results as a course-project demonstration, not field validation.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset" / "dataset.xlsx").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    DEFAULT_DATA_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    create_dataset_profile,
    load_dataset,
    prepare_model_features,
    write_dataset_profile,
)
from src.train import run_training

dataframe, workbook_notes = load_dataset(DEFAULT_DATA_PATH)
print(f"Loaded {len(dataframe):,} records from {DEFAULT_DATA_PATH.name}")

C:\Users\Dell\AppData\Roaming\Python\Python310\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Loaded 139,680 records from dataset.xlsx


## Dataset and target audit

The timestamp is used for chronological ordering and error analysis only. The seven sensor columns are the model inputs. `Keladi` and `Jasmine` are documented as irrigation-event indicators and are not inputs or targets in this experiment.

In [2]:
profile = create_dataset_profile(dataframe, workbook_notes, DEFAULT_DATA_PATH)
print("Shape:", profile["shape"])
print("Time range:", profile["timestamp"]["minimum"], "to", profile["timestamp"]["maximum"])
print("Duplicate rows:", profile["duplicate_full_rows"])
print("Sensor rows with missing values:", profile["sensor_rows_with_any_missing"])
print("Best Time counts:", profile["target_counts"])
print("Mismatches with documented hour rule:", profile["best_time_hour_rule_mismatches"])
print("\nMissing values by column:")
display(
    __import__("pandas").DataFrame.from_dict(profile["columns"], orient="index")
    [["dtype", "missing_count", "missing_percent", "unique_count_including_missing"]]
)
print("\nBest Time definition from workbook Notes:")
print(workbook_notes.get("Best Time (target)", "Definition not found"))

Shape: {'rows': 139680, 'columns': 11}
Time range: 2026-05-12T00:00:00 to 2026-08-16T23:59:00
Duplicate rows: 0
Sensor rows with missing values: 16621
Best Time counts: {'1': 104760, '0': 34920}
Mismatches with documented hour rule: 0

Missing values by column:


,dtype,missing_count,missing_percent,unique_count_including_missing
Timestamp,datetime64[ns],0,0.0000,139680
Soil Moisture 1,float64,2514,1.7998,65
Soil Moisture 2,float64,2514,1.7998,68
Soil Temp1,float64,2514,1.7998,153
Soil Temp2,float64,2514,1.7998,161
Env Temp,float64,2514,1.7998,144
Humidity,float64,2514,1.7998,73
LDR,float64,2514,1.7998,102
Keladi,float64,698,0.4997,3
Jasmine,float64,698,0.4997,3



Best Time definition from workbook Notes:
Target based on the time of collection. 1 = best time (20:00 to 04:59) or another allowed period (05:00 to 09:59 and 16:00 to 19:59, when the farmer is told it is not the best time to water). 0 = irrigation halted (10:00 to 15:59), the worst time because temperatures are highest. No blanks or errors in this column. 104760 rows = 1, 34920 rows = 0.


## Exploration and reproducible model run

The workflow saves a complete dataset profile and charts under `outputs/`. Known sensor reset/failure codes are treated as missing and imputed within a scikit-learn pipeline fitted only on the training period. Ambiguous sensor endpoint values are retained rather than assumed invalid.

In [3]:
summary = run_training(
    data_path=DEFAULT_DATA_PATH,
    model_path=PROJECT_ROOT / "models" / "trained_model.joblib",
    output_path=PROJECT_ROOT / "outputs",
    test_size=0.2,
)
import pandas as pd
display(pd.DataFrame(summary["metrics"]).T.round(4))
print("Chronological holdout:", summary["test_start"], "to", summary["test_end"])
print("Reloaded model sample predictions:", summary["reloaded_model_sample_predictions"])

C:\Users\Dell\AppData\Roaming\Python\Python310\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,accuracy,precision,recall,f1,macro_f1
Most frequent baseline,0.7517,0.7517,1.0000,0.8583,0.4291
Decision tree,0.8101,0.8779,0.8681,0.8730,0.7484


Chronological holdout: 2026-07-28T14:24:00 to 2026-08-16T23:59:00
Reloaded model sample predictions: [0, 0, 1, 0, 1]


## Classification report and error inspection

Precision, recall, and F1 use class `1` as the positive class. The baseline predicts the majority class. Inspect the saved confusion matrix, per-hour error rates, and actual-versus-predicted holdout rows before interpreting the aggregate scores.

In [4]:
import json

report_path = PROJECT_ROOT / "outputs" / "evaluation" / "classification_report.json"
classification_reports = json.loads(report_path.read_text(encoding="utf-8"))
display(pd.DataFrame(classification_reports["Decision tree"]).T.round(3))
predictions = pd.read_csv(PROJECT_ROOT / "outputs" / "evaluation" / "test_predictions.csv")
display(predictions.head(10))

,precision,recall,f1-score,support
0,0.614,0.634,0.624,6936.00
1,0.878,0.868,0.873,21000.00
accuracy,0.810,0.810,0.810,0.81
macro avg,0.746,0.751,0.748,27936.00
weighted avg,0.812,0.810,0.811,27936.00


,Timestamp,actual,predicted,baseline_predicted,error
0,2026-07-28 14:24:00,0,0,1,False
1,2026-07-28 14:25:00,0,0,1,False
2,2026-07-28 14:26:00,0,1,1,True
3,2026-07-28 14:27:00,0,0,1,False
4,2026-07-28 14:28:00,0,1,1,True
5,2026-07-28 14:29:00,0,0,1,False
6,2026-07-28 14:30:00,0,0,1,False
7,2026-07-28 14:31:00,0,0,1,False
8,2026-07-28 14:32:00,0,1,1,True
9,2026-07-28 14:33:00,0,0,1,False


## Saved-model input contract

The following check reloads the serialized model and verifies prediction using exactly the seven documented sensor features. It does not control a pump.

In [5]:
import joblib

artifact = joblib.load(PROJECT_ROOT / "models" / "trained_model.joblib")
assert artifact["feature_columns"] == FEATURE_COLUMNS
model_input = prepare_model_features(dataframe).tail(5)
assert list(model_input.columns) == FEATURE_COLUMNS
predictions_from_saved_model = artifact["model"].predict(model_input)
print("Expected inputs:", artifact["feature_columns"])
print("Predictions:", predictions_from_saved_model.tolist())
print("Target:", TARGET_COLUMN)

Expected inputs: ['Soil Moisture 1', 'Soil Moisture 2', 'Soil Temp1', 'Soil Temp2', 'Env Temp', 'Humidity', 'LDR']
Predictions: [1, 1, 1, 1, 1]
Target: Best Time
